In [39]:
import os
import cv2
import math
import random
import torch
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from facenet_pytorch import InceptionResnetV1
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

In [40]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="8HD54Kwr59F84GmFzQZr")
project = rf.workspace("dataseters").project("face-recognition-gesad-7ha7k")
version = project.version(6)
dataset = version.download("folder")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


loading Roboflow workspace...
loading Roboflow project...


In [41]:
DATASET_PATH = "Face-Recognition-GESAD-6/train"
MODEL_PATH = 'face_landmarker.task'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [42]:
# facenet
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)

# MediaPipe Iris Landmarks
LEFT_IRIS_CENTER = 468
RIGHT_IRIS_CENTER = 473

# mediapipe face mesh
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=True,
    num_faces=1
)
#detector do mediapipe para deteccao do rosto
detector = vision.FaceLandmarker.create_from_options(options)

In [43]:
from mediapipe.tasks.python.vision import drawing_styles, drawing_utils

def get_landmarks_and_image(image_path, draw=True):
    try:
        cv_img = cv2.imread(image_path)
        if cv_img is None:
            return None, None, None

        rgb_img = cv2.cvtColor(cv_img, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_img)

        detection_result = detector.detect(mp_image)

        if len(detection_result.face_landmarks) == 0:
            return cv_img, None, None

        annotated = None
        if draw:
            annotated = np.copy(rgb_img)
            for face_landmarks in detection_result.face_landmarks:
                drawing_utils.draw_landmarks(
                    image=annotated,
                    landmark_list=face_landmarks,
                    connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=drawing_styles.get_default_face_mesh_tesselation_style(),
                )
                drawing_utils.draw_landmarks(
                    image=annotated,
                    landmark_list=face_landmarks,
                    connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=drawing_styles.get_default_face_mesh_contours_style(),
                )
                drawing_utils.draw_landmarks(
                    image=annotated,
                    landmark_list=face_landmarks,
                    connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_LEFT_IRIS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=drawing_styles.get_default_face_mesh_iris_connections_style(),
                )
                drawing_utils.draw_landmarks(
                    image=annotated,
                    landmark_list=face_landmarks,
                    connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_RIGHT_IRIS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=drawing_styles.get_default_face_mesh_iris_connections_style(),
                )

        return cv_img, detection_result.face_landmarks[0], annotated
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None, None, None

In [44]:
def align_face(image, landmarks, target_size=(160, 160)):
    h, w, _ = image.shape

    left_iris = landmarks[LEFT_IRIS_CENTER]
    right_iris = landmarks[RIGHT_IRIS_CENTER]

    lx, ly = int(left_iris.x * w), int(left_iris.y * h)
    rx, ry = int(right_iris.x * w), int(right_iris.y * h)

    dy = ry - ly
    dx = rx - lx
    angle = math.degrees(math.atan2(dy, dx))


    center_x = (lx + rx) // 2
    center_y = (ly + ry) // 2

    M = cv2.getRotationMatrix2D((center_x, center_y), angle, 1.0)
    rotated_img = cv2.warpAffine(image, M, (w, h))


    face_width = int(math.sqrt(dx**2 + dy**2) * 4.0)

    start_x = max(0, center_x - face_width // 2)
    start_y = max(0, center_y - face_width // 2)
    end_x = min(w, center_x + face_width // 2)
    end_y = min(h, center_y + face_width // 2)

    crop = rotated_img[start_y:end_y, start_x:end_x]

    # resize 160x160
    if crop.size == 0: return None
    aligned_face = cv2.resize(crop, target_size)

    return aligned_face


In [45]:
def get_embedding(aligned_face):
    # BGR -> RGB
    aligned_face = cv2.cvtColor(aligned_face, cv2.COLOR_BGR2RGB)

    aligned_face = np.ascontiguousarray(aligned_face, dtype=np.uint8)

    h, w, c = aligned_face.shape
    if c != 3:
        raise ValueError(f"Esperado 3 canais, obtido: {c}")

    face_tensor = torch.from_numpy(aligned_face)
    face_tensor = face_tensor.permute(2, 0, 1).contiguous()
    face_tensor = face_tensor.float().div(255.0)

    face_tensor = (face_tensor - 0.5) / 0.5
    face_tensor = face_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        embedding = resnet(face_tensor).detach().cpu()

    return embedding.flatten().tolist()

In [46]:
print("Starting Dataset Processing...")

X_embeddings = []
y_labels = []
visualize_samples = []
seen_visual_labels = set()


if os.path.exists(DATASET_PATH):
    for root, dirs, files in os.walk(DATASET_PATH):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                full_path = os.path.join(root, file)
                label = os.path.basename(root)

                img_bgr, landmarks, _ = get_landmarks_and_image(full_path)

                if landmarks:
                    aligned_face = align_face(img_bgr, landmarks)

                    if aligned_face is not None:
                        emb = get_embedding(aligned_face)

                        X_embeddings.append(emb)
                        y_labels.append(label)

                        if label not in seen_visual_labels and len(visualize_samples) < 3:
                            visualize_samples.append((aligned_face, label))
                            seen_visual_labels.add(label)

    print(f"Finished. Processed {len(X_embeddings)} faces.")
else:
    print("Dataset path not found. Please check DATASET_PATH.")


Starting Dataset Processing...
Finished. Processed 6173 faces.


In [47]:
if len(X_embeddings) > 0:
    le = LabelEncoder()
    y_encoded = le.fit_transform(y_labels)

    classifier = KNeighborsClassifier(n_neighbors=1, metric='cosine')

    # Validação Cruzada para avaliar a qualidade do modelo
    print("Iniciando Validação Cruzada (5-fold)...")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(classifier, X_embeddings, y_encoded, cv=cv)

    print(f"Acurácia Média (Cross-Val): {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

    # Treino final com 100% dos dados para máxima performance na API
    classifier.fit(X_embeddings, y_encoded)

    print(f"Classifier trained successfully on {len(X_embeddings)} samples (100% of data)!")

    #test and inference

    def recognize_face(image_path):
        print(f"Recognizing: {image_path}")
        img_bgr, landmarks, _ = get_landmarks_and_image(image_path)

        if landmarks is None:
            return "No face detected"

        aligned = align_face(img_bgr, landmarks)
        if aligned is None:
            return "Could not align face"

        emb = get_embedding(aligned)

        prediction_idx = classifier.predict([emb])[0]
        prediction_name = le.inverse_transform([prediction_idx])[0]

        # get cosine distance
        distances, _ = classifier.kneighbors([emb])
        dist = distances[0][0]

        return f"Prediction: {prediction_name} (Distance: {dist:.4f})"



    # random prediction from dataset
    print("\nRandom prediction:")
    all_images = []
    for root, dirs, files in os.walk(DATASET_PATH):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_images.append(os.path.join(root, file))

    if all_images:
        for random_img in range(5):
            test_img = random.choice(all_images)
            print(recognize_face(test_img))
    else:
        print("No images found for random prediction.")

else:
    print("No faces were successfully processed. Check dataset path or MediaPipe task file.")

Iniciando Validação Cruzada (5-fold)...
Acurácia Média (Cross-Val): 0.9997 (+/- 0.0008)
Classifier trained successfully on 6173 samples (100% of data)!

Random prediction:
Recognizing: Face-Recognition-GESAD-6/train\PedroLuna\frame_00335_jpg.rf.6402ed0287ab6a4183c36c0a01e22277.jpg
Prediction: PedroLuna (Distance: 0.0000)
Recognizing: Face-Recognition-GESAD-6/train\AlanBandeira\frame_00531_jpg.rf.140319e3a980e57283af181ab418fd8f.jpg
Prediction: AlanBandeira (Distance: 0.0000)
Recognizing: Face-Recognition-GESAD-6/train\DavidMoreira\frame_01134_jpg.rf.571888334913c50d8a859782980a6465.jpg
Prediction: DavidMoreira (Distance: 0.0000)
Recognizing: Face-Recognition-GESAD-6/train\AlanBandeira\frame_00968_jpg.rf.2c669e54900b98a27d9c9ad54d3e6baa.jpg
Prediction: AlanBandeira (Distance: 0.0000)
Recognizing: Face-Recognition-GESAD-6/train\GuilhermeGaspar\frame_00995_jpg.rf.02ddbd4c13a5f34f4f8d290f890ce9ee.jpg
Prediction: GuilhermeGaspar (Distance: 0.0000)
